# V1-19: OpenAI API 기업 식별 정책 Holdout

18번 local validation에서 조건부 유지한 B 정책을 **변경 없이** 운영 후보 OpenAI API로 한 번 검증한다. 이 노트북은 후보를 비교하거나 Prompt·cutoff를 고르는 단계가 아니다.

고정 정책:

```text
raw exact + fuzzy 후보
→ 고유 exact면 RESOLVED
→ 포괄 그룹명은 NEEDS_CONFIRMATION
→ 그 외에는 ChatOpenAI 검색어 가설 + OpenDART 재검증
→ LLM 후보가 있거나 fuzzy 최고점이 75 이상이면 NEEDS_CONFIRMATION
→ 그렇지 않으면 NOT_FOUND
```

LLM은 검색어를 최대 2개 제안할 뿐이며, LLM이 새로 발견한 후보는 절대로 자동 확정하지 않는다.

## 목적·평가 기준·Holdout 규칙

목적은 실제 운영 후보 `gpt-5.6-luna`가 고정 정책에서 후보 회수를 높이면서 자동 오선택·유효 기업의 NOT_FOUND를 만들지 않는지 확인하는 것이다.

- 05~18번에 사용한 기업·표현·Prompt 예시와 구분한 독립 holdout 36개와, 고정된 그룹 보호 규칙의 회귀 확인 4개를 함께 실행한다.
- 최종 품질 지표는 독립 holdout 36개로만 계산하며, 보호 그룹명 4개는 정책 회귀 결과로 별도 표시한다.
- 기대 정식 기업명·기대 상태·LLM 호출 기대 여부는 실행 전에 고정하고 Prompt에는 전달하지 않는다. 실행 시점에 이 정식명으로 확인한 `corp_code`와 OpenDART 조회 시각을 결과에 기록한다.
- 결과를 본 뒤 Prompt, cutoff, 그룹 목록, 후보 병합 규칙을 바꾸면 이 holdout은 이후 후보의 validation이 된다.
- 자동 오선택 또는 유효 기업의 False Not-found가 한 건이라도 발생하면 최종 채택을 보류한다.

핵심 지표는 Candidate Recall, Unsafe Selection Rate, State Accuracy, Not-found Recall, False Not-found Rate, No-target LLM Candidate Injection Rate다. 보조로 LLM 신규 기여, 구조화 출력, 지연시간, 토큰·추정 비용을 기록한다.

In [ ]:
import io
from datetime import datetime, timezone
from math import ceil
import json
import os
from pathlib import Path
import re
from statistics import median
import sys
import time
from typing import Literal
import xml.etree.ElementTree as ET
import zipfile

import pandas as pd
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from rapidfuzz import fuzz, process

In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')
for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

for required_key in ('OPENDART_API_KEY', 'OPENAI_API_KEY', 'OPENAI_MODEL'):
    if not os.environ.get(required_key):
        raise ValueError(f'.env에 {required_key}를 설정해 주세요.')

api_key = os.environ['OPENDART_API_KEY']
OPENAI_MODEL = os.environ['OPENAI_MODEL']
EXPECTED_OPENAI_MODEL = 'gpt-5.6-luna'
if OPENAI_MODEL != EXPECTED_OPENAI_MODEL:
    raise ValueError(f'이 holdout은 {EXPECTED_OPENAI_MODEL} 전용입니다. 현재 OPENAI_MODEL={OPENAI_MODEL!r}')

TEMPERATURE, REQUEST_TIMEOUT_S, MAX_RETRIES = 0, 300, 0
FUZZY_KEY_LIMIT, FUZZY_RESULT_LIMIT, CUTOFF = 50, 5, 75
MAX_SEARCH_TERMS = 2
OPENAI_INPUT_USD_PER_MTOKEN = 0.20
OPENAI_CACHED_INPUT_USD_PER_MTOKEN = 0.02
OPENAI_OUTPUT_USD_PER_MTOKEN = 1.20

print(f'OpenAI model: {OPENAI_MODEL} / temperature={TEMPERATURE} / reasoning_effort=none')
print(f'API key loaded: {bool(os.environ.get("OPENAI_API_KEY"))} (값은 출력하지 않음)')
print(f'cutoff={CUTOFF} / fuzzy top={FUZZY_RESULT_LIMIT} / max search terms={MAX_SEARCH_TERMS}')
print(f'timeout={REQUEST_TIMEOUT_S}초 / max_retries={MAX_RETRIES}')

In [ ]:
sys.path.insert(0, str(project_root / 'src'))
from dart.client import CORP_CODE_URL, _get_bytes


def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(stream):
        raise RuntimeError(zip_bytes.decode('utf-8', errors='replace'))
    with zipfile.ZipFile(stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])
    root = ET.fromstring(xml_bytes)
    return [{child.tag: child.text or '' for child in item} for item in root.findall('list')]


def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())


def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(values.values()) for key, values in index.items()}


corp_records = load_corp_records(api_key)
name_index = build_name_index(corp_records)
name_keys = list(name_index)
retrieved_at = datetime.now().astimezone().isoformat(timespec='seconds')
print(f'OpenDART 목록 조회 시각: {retrieved_at}')
print(f'법인 레코드: {len(corp_records):,} / 정규화 이름 키: {len(name_keys):,}')

In [ ]:
# Holdout label은 평가 전용이며 LLM prompt에 전달하지 않는다.
# 05~18번과 기업·표현을 구분한 사례로, 실행 후 이 목록을 수정하지 않는다.
CASES = [
    # 고유 정식명·공백/기호 정규화 (8)
    ('한미반도체 매출을 알려줘.', '한미반도체', ['한미반도체'], 'RESOLVED', False, '정식명'),
    ('DB 하이텍 영업이익은?', 'DB 하이텍', ['DB하이텍'], 'RESOLVED', False, '공백/기호'),
    ('HD 현대일렉트릭 실적을 보여줘.', 'HD 현대일렉트릭', ['HD현대일렉트릭'], 'RESOLVED', False, '공백/기호'),
    ('S - OIL 재무정보를 알려줘.', 'S - OIL', ['S-OIL'], 'RESOLVED', False, '공백/기호'),
    ('NC 매출 추세는?', 'NC', ['NC'], 'RESOLVED', False, '정식명'),
    ('한미 약품 영업이익을 알려줘.', '한미 약품', ['한미약품'], 'RESOLVED', False, '공백/기호'),
    ('현대 제철 매출은?', '현대 제철', ['현대제철'], 'RESOLVED', False, '공백/기호'),
    ('대웅 제약 실적을 알려줘.', '대웅 제약', ['대웅제약'], 'RESOLVED', False, '공백/기호'),
    # 새로운 약칭·한글/영문 표기·오타 (12)
    ('디비하이텍 최근 실적을 알려줘.', '디비하이텍', ['DB하이텍'], 'NEEDS_CONFIRMATION', True, '한글 음역'),
    ('에이치디현대일렉트릭 매출을 알려줘.', '에이치디현대일렉트릭', ['HD현대일렉트릭'], 'NEEDS_CONFIRMATION', True, '한글 음역'),
    ('에스오일 영업이익은?', '에스오일', ['S-OIL'], 'NEEDS_CONFIRMATION', True, '한글 음역'),
    ('엔시소프트 실적을 알려줘.', '엔시소프트', ['NC'], 'NEEDS_CONFIRMATION', True, '변경 전 법인명'),
    ('한미약퓸 매출은?', '한미약퓸', ['한미약품'], 'NEEDS_CONFIRMATION', True, '오타'),
    ('현대제첼 최근 실적은?', '현대제첼', ['현대제철'], 'NEEDS_CONFIRMATION', True, '오타'),
    ('대웅제약 영업이익을 알려줘.', '대웅제약', ['대웅제약'], 'RESOLVED', False, '정식명'),
    ('유한양헹 실적을 알려줘.', '유한양헹', ['유한양행'], 'NEEDS_CONFIRMATION', True, '오타'),
    ('코웨이이 매출을 알려줘.', '코웨이이', ['코웨이'], 'NEEDS_CONFIRMATION', True, '오타'),
    ('이마투 실적을 알려줘.', '이마투', ['이마트'], 'NEEDS_CONFIRMATION', True, '오타'),
    ('오뚜깅 영업이익은?', '오뚜깅', ['오뚜기'], 'NEEDS_CONFIRMATION', True, '오타'),
    ('크래프턴 매출 추세는?', '크래프턴', ['크래프톤'], 'NEEDS_CONFIRMATION', True, '오타'),
    # 고정 보호 그룹명 (4): 정책상 LLM 없이 재질문으로 보낸다.
    ('롯데 실적을 알려줘.', '롯데', [], 'NEEDS_CONFIRMATION', False, '보호 그룹명'),
    ('GS 최근 매출을 알려줘.', 'GS', [], 'NEEDS_CONFIRMATION', False, '보호 그룹명'),
    ('신세계 실적을 분석해줘.', '신세계', [], 'NEEDS_CONFIRMATION', False, '보호 그룹명'),
    ('CJ 매출을 알려줘.', 'CJ', [], 'NEEDS_CONFIRMATION', False, '보호 그룹명'),
    # 추가 정식명 (4): holdout의 exact-auto-resolve 경로를 별도로 확인한다.
    ('빙그레 실적을 알려줘.', '빙그레', ['빙그레'], 'RESOLVED', False, '정식명'),
    ('농심 최근 매출을 알려줘.', '농심', ['농심'], 'RESOLVED', False, '정식명'),
    ('오뚜기 실적을 분석해줘.', '오뚜기', ['오뚜기'], 'RESOLVED', False, '정식명'),
    ('HL홀딩스 매출을 알려줘.', 'HL홀딩스', ['HL홀딩스'], 'RESOLVED', False, '정식명'),
    # 대상 부재 (8)
    ('한미반도체2051 실적을 알려줘.', '한미반도체2051', [], 'NOT_FOUND', True, '고유사도 미등록명'),
    ('디비하이텍2051 매출을 알려줘.', '디비하이텍2051', [], 'NOT_FOUND', True, '고유사도 미등록명'),
    ('에스오일2051 매출을 알려줘.', '에스오일2051', [], 'NOT_FOUND', True, '고유사도 미등록명'),
    ('현대제철2051 매출을 알려줘.', '현대제철2051', [], 'NOT_FOUND', True, '고유사도 미등록명'),
    ('코웨이2051 매출을 알려줘.', '코웨이2051', [], 'NOT_FOUND', True, '고유사도 미등록명'),
    ('크래프톤2051 매출을 알려줘.', '크래프톤2051', [], 'NOT_FOUND', True, '고유사도 미등록명'),
    ('푸른바다테크2042 실적을 알려줘.', '푸른바다테크2042', [], 'NOT_FOUND', True, '가상명'),
    ('제타오로라홀딩스2051 매출을 알려줘.', '제타오로라홀딩스2051', [], 'NOT_FOUND', True, '가상명'),
    # 기타 경계 사례 (4)
    ('한국항공우주산업 최근 실적을 알려줘.', '한국항공우주산업', ['한국항공우주'], 'NEEDS_CONFIRMATION', True, '변경 전 법인명'),
    ('오리온홀딩스 실적을 알려줘.', '오리온홀딩스', ['오리온홀딩스'], 'RESOLVED', False, '지주사 정식명'),
    ('신라면 매출을 알려줘.', '신라면', [], 'NEEDS_CONFIRMATION', True, '브랜드명'),
    ('현대엔지니어링 실적을 알려줘.', '현대엔지니어링', [], 'NEEDS_CONFIRMATION', True, '동일 법인명'),
]
POLICY_REGRESSION_TERMS = {'롯데', 'GS', '신세계', 'CJ'}
TEST_CASES = [
    {'질문': question, '원문 기업 표현': raw, '기대 대상': targets, '기대 상태': state, 'LLM 호출 기대': expected_llm, '유형': category,
     '평가 구분': '정책 회귀' if raw in POLICY_REGRESSION_TERMS else '독립 holdout'}
    for question, raw, targets, state, expected_llm, category in CASES
]

TARGETS = sorted({target for case in TEST_CASES for target in case['기대 대상']})
target_records, ambiguous_target_rows = {}, []
for target in TARGETS:
    matches = [record for record in corp_records if normalize_name(record['corp_name']) == normalize_name(target)]
    if len(matches) == 1:
        target_records[target] = matches[0]
    elif not matches:
        ambiguous_target_rows.append({'기대 정식명': target, 'OpenDART 법인명': '(정규화 exact 0건)', 'corp_code': '-', 'stock_code': '-'})
    else:
        ambiguous_target_rows.extend({
            '기대 정식명': target, 'OpenDART 법인명': record['corp_name'], 'corp_code': record['corp_code'],
            'stock_code': record.get('stock_code') or '(비상장/미기재)'
        } for record in matches)

if ambiguous_target_rows:
    display(pd.DataFrame(ambiguous_target_rows))
    raise AssertionError('정답 정식명이 OpenDART에서 0건 또는 복수 법인에 매핑됩니다. 위 목록을 확인해 holdout 사례를 단일 법인명으로 교체한 뒤 다시 실행하세요.')

for case in TEST_CASES:
    if case['기대 상태'] == 'NOT_FOUND' and name_index.get(normalize_name(case['원문 기업 표현'])):
        raise AssertionError(f"{case['원문 기업 표현']}: 미등록 holdout 사례 exact 검증 실패")

test_case_frame = pd.DataFrame(TEST_CASES)
print(f'독립 holdout: {(test_case_frame["평가 구분"] == "독립 holdout").sum()}개 / 정책 회귀: {(test_case_frame["평가 구분"] == "정책 회귀").sum()}개 / 정답 기업: {len(TARGETS)}개')
display(test_case_frame[['원문 기업 표현', '평가 구분', '유형', '기대 상태', 'LLM 호출 기대', '기대 대상']])

In [ ]:
def exact_candidates(query, source='raw_exact'):
    return {record['corp_code']: {'record': record, 'sources': {source}, 'score': None} for record in name_index.get(normalize_name(query), [])}


def fuzzy_candidates(query, source='raw_fuzzy'):
    scored = process.extract(normalize_name(query), name_keys, scorer=fuzz.ratio, limit=FUZZY_KEY_LIMIT)
    candidates = {}
    for key, score, _ in scored:
        for record in name_index[key]:
            current = candidates.get(record['corp_code'])
            if current is None or score > current['score']:
                candidates[record['corp_code']] = {'record': record, 'sources': {source}, 'score': score}
    return dict(sorted(candidates.items(), key=lambda item: (item[1]['score'], bool(item[1]['record']['stock_code'].strip()), item[1]['record']['modify_date']), reverse=True)[:FUZZY_RESULT_LIMIT])


def merge_candidates(*candidate_sets):
    merged = {}
    for candidate_set in candidate_sets:
        for corp_code, candidate in candidate_set.items():
            existing = merged.setdefault(corp_code, {'record': candidate['record'], 'sources': set(), 'score': candidate['score']})
            existing['sources'].update(candidate['sources'])
            if candidate['score'] is not None:
                existing['score'] = max(existing['score'] or 0, candidate['score'])
    return merged


# 18번에서 고정한 포괄 그룹 보호 목록이다. holdout 결과를 보고 수정하지 않는다.
GENERIC_GROUP_TERMS = {normalize_name(name) for name in ('롯데', 'GS', '신세계', 'CJ')}


def candidate_display(candidates):
    return [f"{item['record']['corp_name']} ({item['record']['stock_code'].strip() or '비상장'})" for item in candidates.values()]

In [ ]:
class EntitySearchPlan(BaseModel):
    decision: Literal['candidate', 'ambiguous', 'unknown'] = Field(description='검색 가설 제안, 포괄·다의적 표현, 또는 해석 불가 상태')
    search_terms: list[str] = Field(default_factory=list, max_length=MAX_SEARCH_TERMS, description='OpenDART에서 검증할 정식 기업명 검색 가설. corp_code·종목코드·설명 문장은 금지.')


DETAILED_PROMPT = '''
역할: 당신은 한국 기업 리서치 시스템의 'OpenDART 기업명 검색어 계획기'다. 기업을 확정하거나 corp_code를 고르는 역할이 아니다. 프로그램은 원문 기업 표현의 OpenDART exact 결과가 고유하지 않을 때만 호출한다.

목표: 사용자 질문의 원문 기업 표현을 OpenDART 기업 목록에서 다시 찾기 위한 검색 가설을 최대 2개 만든다. 제안은 후보 탐색 입력일 뿐이며, 뒤의 프로그램이 OpenDART 목록으로 존재 여부를 검증하고 필요하면 사용자에게 재질문한다.

candidate를 선택할 조건:
1. 원문이 특정 기업을 가리킨다고 볼 만한 널리 쓰이는 약칭·축약·브랜드식 표기인 경우, 그 기업의 정식 법인명 또는 공식 영문 표기를 검색 가설로 제안한다.
2. 띄어쓰기, 대소문자, 한글 음역/영문 공식 표기 차이, 한두 글자의 명백한 오타처럼 표면 형태만 달라도 특정 기업을 가리킬 근거가 충분한 경우, OpenDART에 있을 법한 표준 기업명으로 바꿔 제안한다.
3. 원문을 그대로 반복하는 것은 그 표현 자체가 정식 기업명 검색어로 유효할 가능성이 있을 때만 허용한다. 별칭·오타·음역 표기라면 같은 문자열을 반복하지 말고 표준 검색 가설을 제안한다.

ambiguous 또는 unknown을 선택할 조건:
1. 그룹명, 업종명, 보통명사, 한 단어가 여러 독립 기업을 가리킬 수 있는 표현처럼 특정 법인을 지목할 근거가 부족하면 ambiguous와 빈 목록을 반환한다.
2. 존재하지 않는 것으로 보이거나 임의 문자열이며 특정 기업으로 연결할 근거가 없으면 unknown과 빈 목록을 반환한다.

금지 규칙:
- 사용자가 언급하지 않은 계열사, 경쟁사, 유사 업종 회사를 연상으로 추가하지 않는다.
- 확신이 없다는 이유로 그럴듯한 실제 기업명을 임의로 만들거나, raw 표현과 무관한 회사를 제안하지 않는다.
- corp_code, 종목코드, 거래소 코드, 투자 의견, 근거 설명, 문장을 search_terms에 넣지 않는다.
- 최종 선택·순위·확정이라는 표현을 하지 않는다.

출력 계약: decision은 candidate, ambiguous, unknown 중 하나다. candidate일 때만 공백을 제외한 기업명 검색어를 1~2개 반환한다. ambiguous와 unknown은 반드시 search_terms=[]를 반환한다. 설명 없이 JSON 객체만 반환한다.
'''

openai_llm = ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE, reasoning_effort='none', timeout=REQUEST_TIMEOUT_S, max_retries=MAX_RETRIES)
openai_planner = openai_llm.with_structured_output(EntitySearchPlan, method='json_schema', include_raw=True)


def clean_terms(plan):
    if plan is None or plan.decision != 'candidate':
        return []
    terms, seen = [], set()
    for raw_term in plan.search_terms:
        term, key = raw_term.strip(), normalize_name(raw_term)
        if term and key and not key.isdigit() and key not in seen and len(term) <= 80:
            terms.append(term)
            seen.add(key)
    return terms[:MAX_SEARCH_TERMS]


def validate_output_contract(plan):
    terms = clean_terms(plan)
    if plan.decision == 'candidate' and not terms:
        return terms, 'output_contract_error: candidate인데 유효 검색어가 없음'
    if plan.decision != 'candidate' and plan.search_terms:
        return terms, 'output_contract_error: non-candidate인데 검색어가 있음'
    return terms, None


def usage_from_result(result):
    raw = result.get('raw') if isinstance(result, dict) else None
    usage = getattr(raw, 'usage_metadata', None) or {}
    metadata = getattr(raw, 'response_metadata', None) or {}
    token_usage = metadata.get('token_usage') or {}
    input_tokens = usage.get('input_tokens', token_usage.get('prompt_tokens'))
    output_tokens = usage.get('output_tokens', token_usage.get('completion_tokens'))
    total_tokens = usage.get('total_tokens', token_usage.get('total_tokens'))
    cached_input_tokens = (usage.get('input_token_details') or {}).get('cache_read', 0) or 0
    actual_model = metadata.get('model_name') or metadata.get('model') or getattr(raw, 'model_name', None)
    return input_tokens, output_tokens, total_tokens, cached_input_tokens, actual_model


def estimate_openai_cost(input_tokens, cached_input_tokens, output_tokens):
    if input_tokens is None or output_tokens is None:
        return None
    input_tokens = int(input_tokens)
    cached_input_tokens = min(max(int(cached_input_tokens or 0), 0), input_tokens)
    return ((input_tokens - cached_input_tokens) * OPENAI_INPUT_USD_PER_MTOKEN + cached_input_tokens * OPENAI_CACHED_INPUT_USD_PER_MTOKEN + int(output_tokens) * OPENAI_OUTPUT_USD_PER_MTOKEN) / 1_000_000


def expand_with_openai(question, raw_name):
    started_at = time.perf_counter()
    try:
        result = openai_planner.invoke([('system', DETAILED_PROMPT), ('human', f'사용자 질문: {question}\n원문 기업 표현: {raw_name}')])
        plan = result.get('parsed')
        parsing_error = result.get('parsing_error')
        usage = usage_from_result(result)
        if plan is None:
            error = f'parsing_error: {parsing_error}' if parsing_error else 'parsed=None'
            return None, [], {}, error, (time.perf_counter() - started_at) * 1_000, usage
        terms, contract_error = validate_output_contract(plan)
        if contract_error:
            return plan, terms, {}, contract_error, (time.perf_counter() - started_at) * 1_000, usage
        candidates = merge_candidates(*(merge_candidates(exact_candidates(term, 'llm_exact'), fuzzy_candidates(term, 'llm_fuzzy')) for term in terms))
        return plan, terms, candidates, None, (time.perf_counter() - started_at) * 1_000, usage
    except Exception as exc:
        return None, [], {}, f'{type(exc).__name__}: {exc}', (time.perf_counter() - started_at) * 1_000, (None, None, None, 0, None)

In [ ]:
# holdout 사례와 무관한 구조화 출력·인증 확인 호출이다. 비용은 평가 호출과 분리해 기록한다.
smoke_rows = []
print('[사전 호출] OpenAI API / 비-holdout 입력 / 호출 시작', flush=True)
started_at = time.perf_counter()
try:
    smoke_result = openai_planner.invoke([('system', DETAILED_PROMPT), ('human', '사용자 질문: 테스트입니다.\n원문 기업 표현: 연결확인테스트법인')])
    smoke_plan = smoke_result.get('parsed')
    smoke_error = smoke_result.get('parsing_error')
    _, smoke_contract_error = validate_output_contract(smoke_plan) if smoke_plan else ([], 'parsed=None')
    smoke_error = smoke_error or smoke_contract_error
    smoke_usage = usage_from_result(smoke_result)
    smoke_cost = estimate_openai_cost(smoke_usage[0], smoke_usage[3], smoke_usage[1])
    smoke_rows.append({'상태': '성공' if smoke_plan is not None and smoke_error is None else '파싱 실패', '응답 모델': smoke_usage[4], '입력 토큰': smoke_usage[0], '캐시 입력 토큰': smoke_usage[3], '출력 토큰': smoke_usage[1], '사전 호출 추정 비용(USD)': smoke_cost, '오류': str(smoke_error) if smoke_error else None})
except Exception as exc:
    smoke_rows.append({'상태': '호출 실패', '응답 모델': None, '입력 토큰': None, '캐시 입력 토큰': None, '출력 토큰': None, '사전 호출 추정 비용(USD)': None, '오류': f'{type(exc).__name__}: {exc}'})

print(f'[사전 호출] 완료 / {(time.perf_counter() - started_at) * 1_000:,.0f}ms', flush=True)
display(pd.DataFrame(smoke_rows))
SMOKE_OK = smoke_rows[0]['상태'] == '성공'
if not SMOKE_OK:
    print('사전 호출에 실패했습니다. 다음 평가 셀은 실행하지 마세요. 인증·모델 접근·구조화 출력 계약을 먼저 해결해야 품질 holdout으로 해석할 수 있습니다.')

In [ ]:
def resolve_case(case):
    missing_targets = set(case['기대 대상']) - set(target_records)
    if missing_targets:
        raise RuntimeError(f'정답 법인 사전검증이 완료되지 않았습니다: {sorted(missing_targets)}. 사례 정의 셀부터 다시 실행하세요.')

    started_at = time.perf_counter()
    raw_name = case['원문 기업 표현']
    raw_exact = exact_candidates(raw_name)
    raw_fuzzy = fuzzy_candidates(raw_name)
    baseline = merge_candidates(raw_exact, raw_fuzzy)
    best_fuzzy_score = max((candidate['score'] or 0 for candidate in raw_fuzzy.values()), default=0)
    key = normalize_name(raw_name)
    llm_called, plan, terms, llm_candidates, llm_error, llm_latency_ms = False, None, [], {}, None, None
    input_tokens, output_tokens, total_tokens, cached_input_tokens, actual_model = None, None, None, 0, None
    fallback_used = False

    if key in GENERIC_GROUP_TERMS:
        state, selected_code, candidates = 'NEEDS_CONFIRMATION', None, baseline
    elif len(raw_exact) == 1:
        state, selected_code, candidates = 'RESOLVED', next(iter(raw_exact)), baseline
    else:
        llm_called = True
        plan, terms, llm_candidates, llm_error, llm_latency_ms, usage = expand_with_openai(case['질문'], raw_name)
        input_tokens, output_tokens, total_tokens, cached_input_tokens, actual_model = usage
        candidates = merge_candidates(baseline, llm_candidates)
        fallback_used = llm_error is not None
        if llm_candidates:
            state, selected_code = 'NEEDS_CONFIRMATION', None
        elif best_fuzzy_score >= CUTOFF:
            state, selected_code = 'NEEDS_CONFIRMATION', None
        else:
            state, selected_code = 'NOT_FOUND', None

    expected_codes = {target_records[target]['corp_code'] for target in case['기대 대상']}
    baseline_codes, candidate_codes = set(baseline), set(candidates)
    policy_latency_ms = (time.perf_counter() - started_at) * 1_000
    estimated_cost = estimate_openai_cost(input_tokens, cached_input_tokens, output_tokens)
    return {
        **case, '상태': state, '선택 corp_code': selected_code, '내부 후보 풀': candidate_display(candidates), '내부 후보 수': len(candidates),
        '최고 fuzzy 점수': best_fuzzy_score, 'LLM 호출': llm_called, 'LLM 호출 기대 일치': llm_called == case['LLM 호출 기대'], 'LLM 결정': plan.decision if plan else None,
        'LLM 검색어': terms, 'LLM 오류': llm_error, 'LLM fallback 사용': fallback_used, 'LLM 시간(ms)': llm_latency_ms, '정책 시간(ms)': policy_latency_ms,
        '입력 토큰': input_tokens, '캐시 입력 토큰': cached_input_tokens, '출력 토큰': output_tokens, '총 토큰': total_tokens, '응답 모델': actual_model, '추정 비용(USD)': estimated_cost,
        '정답 corp_code': sorted(expected_codes), '정답 후보 포함': bool(expected_codes & candidate_codes) if expected_codes else None,
        '기준 후보 정답 포함': bool(expected_codes & baseline_codes) if expected_codes else None,
        'LLM 신규 기여': bool((expected_codes & set(llm_candidates)) - baseline_codes) if expected_codes else None,
        '자동 확정 정답': state == 'RESOLVED' and selected_code in expected_codes,
        '위험 자동 확정': state == 'RESOLVED' and (case['기대 상태'] != 'RESOLVED' or selected_code not in expected_codes),
        '상태 정답': state == case['기대 상태'],
        '미등록 LLM 후보 주입': case['기대 상태'] == 'NOT_FOUND' and bool(llm_candidates),
    }


if not SMOKE_OK:
    raise RuntimeError('사전 호출 실패: 품질 holdout API 호출을 중단합니다.')

rows = []
for case_number, case in enumerate(TEST_CASES, start=1):
    print(f'[{case_number}/{len(TEST_CASES)}] {case["원문 기업 표현"]!r} / API 호출 여부 판단', flush=True)
    rows.append(resolve_case(case))

results = pd.DataFrame(rows)
print(f'완료: 사례 {len(results)}개 / API LLM 호출 {int(results["LLM 호출"].sum())}회', flush=True)
display(results[['원문 기업 표현', '유형', '기대 상태', '상태', '정답 후보 포함', 'LLM 호출', 'LLM 검색어', 'LLM 오류']])

In [ ]:
def p95(values):
    return sorted(values)[ceil(len(values) * 0.95) - 1] if values else None


holdout_results = results[results['평가 구분'] == '독립 holdout'].copy()
policy_regression_results = results[results['평가 구분'] == '정책 회귀'].copy()
positive = holdout_results[holdout_results['기대 대상'].map(bool)]
no_target = holdout_results[holdout_results['기대 상태'] == 'NOT_FOUND']
ambiguous = holdout_results[holdout_results['기대 상태'] == 'NEEDS_CONFIRMATION']
auto = holdout_results[holdout_results['상태'] == 'RESOLVED']
llm_rows = holdout_results[holdout_results['LLM 호출']]
latencies = holdout_results['정책 시간(ms)'].tolist()
costs = [value for value in holdout_results['추정 비용(USD)'] if value is not None and not pd.isna(value)]
smoke_costs = [value for value in pd.DataFrame(smoke_rows)['사전 호출 추정 비용(USD)'] if value is not None and not pd.isna(value)]

summary = pd.DataFrame([ {
    'Candidate Recall': positive['정답 후보 포함'].mean(),
    'Auto-resolution Success Rate': positive['자동 확정 정답'].mean(),
    'Unsafe Selection Rate': auto['위험 자동 확정'].mean() if len(auto) else 0.0,
    'State Accuracy': holdout_results['상태 정답'].mean(),
    'Abstention Accuracy': ambiguous['상태 정답'].mean() if len(ambiguous) else None,
    'Not-found Recall': (no_target['상태'] == 'NOT_FOUND').mean(),
    'False Not-found Rate': (positive['상태'] == 'NOT_FOUND').mean(),
    'No-target LLM Candidate Injection Rate': no_target['미등록 LLM 후보 주입'].mean(),
    'LLM Expansion Contribution': positive['LLM 신규 기여'].mean(),
    'LLM Candidate Recall': llm_rows[llm_rows['기대 대상'].map(bool)]['정답 후보 포함'].mean(),
    'Output Contract Validity': llm_rows['LLM 오류'].isna().mean() if len(llm_rows) else None,
    'LLM Call Rate': holdout_results['LLM 호출'].mean(),
    'LLM Call-path Accuracy': holdout_results['LLM 호출 기대 일치'].mean(),
    'LLM fallback Rate': llm_rows['LLM fallback 사용'].mean() if len(llm_rows) else None,
    '입력 기준 fallback 발생률': holdout_results['LLM fallback 사용'].mean(),
    '내부 후보 풀 중앙값': holdout_results['내부 후보 수'].median(),
    'p50 ms': median(latencies), 'p95 ms': p95(latencies),
    '평가 호출 추정 비용(USD)': sum(costs) if costs else None,
    '사전 호출 추정 비용(USD)': sum(smoke_costs) if smoke_costs else None,
    '노트북 총 추정 비용(USD)': (sum(costs) if costs else 0) + (sum(smoke_costs) if smoke_costs else 0) if costs or smoke_costs else None,
} ])
display(summary.T.rename(columns={0: '독립 OpenAI API holdout (36개)'}))

print('정책 회귀 결과(독립 holdout 지표에서는 제외):')
display(policy_regression_results[['원문 기업 표현', '기대 상태', '상태', 'LLM 호출', 'LLM 호출 기대 일치', '상태 정답']])

display(holdout_results[holdout_results['LLM 호출']][['원문 기업 표현', '유형', 'LLM 결정', 'LLM 검색어', '정답 후보 포함', 'LLM 신규 기여', '미등록 LLM 후보 주입', 'LLM 오류', '추정 비용(USD)']])

## Holdout 해석 규칙

- Unsafe Selection Rate가 0이 아니거나 False Not-found Rate가 0보다 크면 최종 정책 채택을 보류한다.
- Candidate Recall이 낮거나 LLM 신규 기여 없이 무관 후보 주입만 늘리면 API LLM 확장 채택을 보류한다.
- Not-found Recall과 Candidate Recall이 충돌하면 한쪽을 희생해 즉시 조정하지 않고, 실패 사례와 사용자 영향으로 별도 결정한다.
- holdout 결과를 본 뒤 Prompt·cutoff·그룹 목록·후보 병합 규칙을 변경하지 않는다. 변경이 필요하면 이 결과는 validation으로 기록하고 새 holdout을 만든다.

실행 결과를 저장한 뒤 알려주면, 19번 HTML 보고서에 상단 요약(목적·핵심 결과·판단)과 하단 상세 결과를 기록한다.